# 02 — Prompt Engineering: From Minimal to Production-Grade

**Goal**: see how prompt structure affects the quality of financial analysis
output from the same model with the same retrieved context.

We keep the **retrieved context fixed** (a hardcoded sample so no DB is
needed) and iterate on the **prompt only**.  This isolates the effect of
prompt design from the effect of retrieval quality.

**Prerequisites**:
```bash
# Local Ollama model (no API keys required)
ollama pull qwen2.5:14b-instruct
```

Optional — to test with paid tiers, set `OPENAI_API_KEY` or
`ANTHROPIC_API_KEY` in `.env` and change `TIER` in the setup cell.

In [ ]:
import time
import textwrap

from sky_finance.strategies.engine import run_with_model

TIER   = 'local'   # 'local' | 'nano' | 'advanced' | 'claude'
TICKER = 'AAPL'

# --- helper: run and pretty-print -------------------------------------------
def run(system: str, user: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    text, model_id, usage = run_with_model(TIER, system, user)
    elapsed = round(time.perf_counter() - t0, 1)
    return text, elapsed

def show(label: str, text: str, elapsed: float) -> None:
    sep = '=' * 68
    print(f'\n{sep}')
    print(f'  {label}  ({elapsed}s)')
    print(sep)
    for line in text.strip().splitlines():
        print(textwrap.fill(line, width=68) if len(line) > 68 else line)

print(f'Tier: {TIER}  Ticker: {TICKER}')

In [ ]:
# Fixed RAG context — identical for all four prompt versions below.
# In production this comes from the pgvector similarity search in notebook 01.
CONTEXT = '''
### Apple Q4 Earnings Beat [positive] (sim=0.91)
Apple reported Q4 revenue of $94.9B, beating consensus estimates by 3.2 %.
iPhone revenue reached $46.2B on strong demand from China and India.
Services revenue hit a new record at $24.2B, up 16 % YoY.
EPS of $1.46 beat the $1.39 estimate; gross margin expanded 60 bps to 46.2 %.

### iPhone Supply Chain Risk [negative] (sim=0.87)
TSMC Arizona fab faces equipment installation delays, putting 15 % of iPhone
production capacity at risk through H1 next year. Apple is qualifying suppliers
in Vietnam and India as backup, but the transition could take 12-18 months.
Foxconn warned of component shortfalls affecting 3-5 % of annual volume.

### Fed Rate Decision Impact [neutral] (sim=0.71)
The Federal Reserve held rates at 5.25-5.5 % for the fourth consecutive meeting.
Technology stocks remain sensitive to rates as future earnings are discounted
at higher rates. Apple's $90B annual buyback programme benefits from lower
borrowing costs. Analysts await forward guidance on the pace of future cuts.
'''.strip()

print('Context loaded:', len(CONTEXT), 'characters')

## Prompt v1 — Minimal

No context, no structure.  Just ask.

**What to expect**: the model draws on training data (months out of date) and
produces generic, unverifiable claims.  Hallucination risk is high.

In [ ]:
SYS_V1 = 'You are a financial analyst.'
USR_V1 = f'Analyze {TICKER} stock.'

text_v1, t_v1 = run(SYS_V1, USR_V1)
show('v1 — MINIMAL  (no context, no structure)', text_v1, t_v1)

## Prompt v2 — Context + Structure

We inject the retrieved news context and ask for a specific structure.

**What to expect**: output is now *grounded in real evidence*, but without a
persona or reasoning chain the model still tends to summarise rather than
analyse.

In [ ]:
SYS_V2 = 'You are a financial analyst. Be concise and data-driven.'
USR_V2 = f'''Analyze {TICKER} based on the following recent news.

{CONTEXT}

Provide:
1. Key risks
2. Key opportunities
3. One-sentence outlook'''

text_v2, t_v2 = run(SYS_V2, USR_V2)
show('v2 — CONTEXT + STRUCTURE', text_v2, t_v2)

## Prompt v3 — Persona + Chain-of-Thought + Format

Three additions over v2:

1. **Persona** — the analyst works at a hedge fund; their output influences
   real position-sizing decisions.  This shifts tone from academic to
   actionable.

2. **Chain-of-thought** — explicit step-by-step reasoning forces the model
   to weigh evidence before concluding, reducing hallucination.

3. **Strict output format** — predictable structure makes the output easy to
   parse, compare, and store in `strategy_results`.

This is essentially the production prompt sky-finance uses.

In [ ]:
SYS_V3 = '''You are a quantitative equity analyst at a long/short hedge fund.
Your analysis is read by portfolio managers making position-sizing decisions.
Be specific, cite evidence from the context, and quantify where possible.
Do not invent facts not present in the provided context.'''

USR_V3 = f'''Analyze {TICKER} using ONLY the evidence in the context below.

Think through this step by step:
1. Identify the strongest bullish signals and their magnitude.
2. Identify the strongest bearish / risk signals.
3. Assess how these signals balance against each other.
4. Form a near-term directional view.

Context:
{CONTEXT}

Output format — fill in each field:
**Signal**: [Strong Bull / Bull / Neutral / Bear / Strong Bear]
**Bull case**: [2 specific evidence-backed points]
**Bear case**: [2 specific evidence-backed points]
**Outlook**: [2 sentences max, cite specific data points]
**Confidence**: [High / Medium / Low] — [one-line reason]'''

text_v3, t_v3 = run(SYS_V3, USR_V3)
show('v3 — PERSONA + CoT + FORMAT  (production-grade)', text_v3, t_v3)

## Side-by-side comparison

In [ ]:
rows = [
    ('v1 — minimal',           t_v1, 'no context → draws on training data; high hallucination risk'),
    ('v2 — context+structure', t_v2, 'grounded in evidence; summarises rather than analyses'),
    ('v3 — production-grade',  t_v3, 'persona + CoT forces evidence-backed, quantified reasoning'),
]

print('\nCOMPARISON SUMMARY')
print('=' * 68)
for label, elapsed, note in rows:
    print(f'\n  {label}  ({elapsed}s)')
    print(f'  → {note}')

## Key Takeaways

1. **Minimal prompts produce minimal output** — v1 draws on training data
   that may be months out of date and cannot be verified.

2. **Context alone is not enough** — v2 is grounded in real news, but without
   a reasoning structure the model summarises rather than analyses.

3. **Persona + CoT + format = production quality** — v3 forces the model to:
   - Adopt a perspective where output *matters* (reduces vagueness)
   - Think through evidence before concluding (reduces hallucination)
   - Emit a predictable, parse-able structure (enables storage + comparison)

4. **This is exactly what sky-finance does in production** —
   `strategies.prompt_template` (stored in the DB) is the system prompt, and
   `strategies.rag_query_template` drives the context retrieval.  Iterate on
   both via the dashboard at `/strategies`, no code changes needed.

5. **The cheapest single improvement**: add a format specification.  Even a
   weak local model produces dramatically better, more parseable output when
   it knows exactly what shape the answer should take.